# 🧪 Sesión 25 · Consultas Complejas, Joins y Carga Ansiosa (Eager Loading)

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Por qué una lista de 100 autores dispara 101 consultas?

---

## 🖥️ Así se verá el resultado final

```http
GET /api/autores
HTTP/1.1 200 OK
{"autores":[
  {"id":1,"nombre":"Gabriel García Márquez",
   "libros":[{"id":1,"titulo":"Cien años de soledad"}]},
  {"id":2,"nombre":"Julio Cortázar",
   "libros":[{"id":2,"titulo":"Rayuela"}]}
]}

GET /api/autores/1
HTTP/1.1 200 OK
{"id":1,"nombre":"Gabriel García Márquez",
 "libros":[{"id":1,"titulo":"Cien años de soledad"}]}
```

_El resultado es el mismo; la diferencia es cuántas consultas hiciste para lograrlo._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Cargar relaciones con selectinload.
- Evitar el problema N+1.
- Verificar que la respuesta anida los libros del autor.

## ✅ Criterios de aceptación

El verificador ejecutará los casos HTTP y comprobará los archivos del contrato. Todo debe quedar en verde:

- [ ] Los autores llegan con sus libros → HTTP 200
- [ ] Un autor concreto trae sus libros → HTTP 200
- [ ] El listado de libros sigue disponible → HTTP 200
- [ ] Se usa carga ansiosa

## 🧩 El reto

Implementa una consulta de autores con sus libros usando carga ansiosa para evitar el problema N+1.

**Requisitos del contrato**

- Siembra exactamente: Gabriel García Márquez → «Cien años de soledad»; Julio Cortázar → «Rayuela».
- GET /api/autores devuelve todos los autores con sus libros anidados.
- GET /api/autores/{id} devuelve un autor con sus libros.
- La consulta usa selectinload o joinedload.
- La respuesta no incluye autores sin sus libros cuando existen.
- El listado de libros sigue disponible en /api/libros.

> **Pista conceptual:** Sin carga ansiosa, cada autor dispara una consulta para sus libros. `selectinload(Autor.libros)` resuelve la relación en una sola consulta adicional.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion25.md` explicando el problema N+1.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [2]:

%%writefile app.py

from fastapi import FastAPI, HTTPException
from sqlalchemy import create_engine, Column, Integer, String, ForeignKey, select
from sqlalchemy.orm import declarative_base, relationship, sessionmaker, selectinload

# --------------------------------------------------
# 1. CONFIGURACIÓN DE FASTAPI Y BASE DE DATOS
# --------------------------------------------------

app = FastAPI(
    title="Biblioteca API",
    version="1.0.0"
)

DATABASE_URL = "sqlite:///./biblioteca.db"

engine = create_engine(
    DATABASE_URL,
    connect_args={"check_same_thread": False}
)

SessionLocal = sessionmaker(
    bind=engine,
    autoflush=False,
    autocommit=False
)

Base = declarative_base()


# --------------------------------------------------
# 2. MODELOS DE LA BASE DE DATOS
# --------------------------------------------------

class Autor(Base):
    __tablename__ = "autores"

    id = Column(Integer, primary_key=True, index=True)
    nombre = Column(String, nullable=False)

    libros = relationship(
        "Libro",
        back_populates="autor"
    )


class Libro(Base):
    __tablename__ = "libros"

    id = Column(Integer, primary_key=True, index=True)
    titulo = Column(String, nullable=False)

    autor_id = Column(
        Integer,
        ForeignKey("autores.id"),
        nullable=False
    )

    autor = relationship(
        "Autor",
        back_populates="libros"
    )


# --------------------------------------------------
# 3. CREAR TABLAS Y CARGAR DATOS INICIALES
# --------------------------------------------------

def inicializar_bd():

    Base.metadata.create_all(bind=engine)

    with SessionLocal() as db:

        existe = db.execute(
            select(Autor.id).limit(1)
        ).first()

        if existe is None:

            autor1 = Autor(
                id=1,
                nombre="Gabriel García Márquez",
                libros=[
                    Libro(
                        id=1,
                        titulo="Cien años de soledad"
                    )
                ]
            )

            autor2 = Autor(
                id=2,
                nombre="Julio Cortázar",
                libros=[
                    Libro(
                        id=2,
                        titulo="Rayuela"
                    )
                ]
            )

            db.add_all([autor1, autor2])
            db.commit()


# Inicializar al cargar la aplicación
inicializar_bd()


# --------------------------------------------------
# 4. CONSULTAR TODOS LOS AUTORES CON SUS LIBROS
# --------------------------------------------------

@app.get("/api/autores")
def listar_autores():

    with SessionLocal() as db:

        consulta = (
            select(Autor)
            .options(selectinload(Autor.libros))
            .order_by(Autor.id)
        )

        autores = db.execute(consulta).scalars().all()

        return {
            "autores": [
                {
                    "id": autor.id,
                    "nombre": autor.nombre,
                    "libros": [
                        {
                            "id": libro.id,
                            "titulo": libro.titulo
                        }
                        for libro in autor.libros
                    ]
                }
                for autor in autores
            ]
        }


# --------------------------------------------------
# 5. CONSULTAR UN AUTOR ESPECÍFICO CON SUS LIBROS
# --------------------------------------------------

@app.get("/api/autores/{id}")
def obtener_autor(id: int):

    with SessionLocal() as db:

        consulta = (
            select(Autor)
            .options(selectinload(Autor.libros))
            .where(Autor.id == id)
        )

        autor = db.execute(consulta).scalars().first()

        if autor is None:
            raise HTTPException(
                status_code=404,
                detail="Autor no encontrado"
            )

        return {
            "id": autor.id,
            "nombre": autor.nombre,
            "libros": [
                {
                    "id": libro.id,
                    "titulo": libro.titulo
                }
                for libro in autor.libros
            ]
        }


# --------------------------------------------------
# 6. CONSULTAR TODOS LOS LIBROS
# --------------------------------------------------

@app.get("/api/libros")
def listar_libros():

    with SessionLocal() as db:

        libros = db.execute(
            select(Libro).order_by(Libro.id)
        ).scalars().all()

        return {
            "libros": [
                {
                    "id": libro.id,
                    "titulo": libro.titulo
                }
                for libro in libros
            ]
        }


Writing app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Utilizar Python 3.13, FastAPI y SQLAlchemy en Google Colab, trabajando con una base de datos SQLite que almacena información de autores y libros.|
| **Rol** | Actuar como un desarrollador de software especializado en APIs RESTful, bases de datos y SQLAlchemy, encargado de crear consultas eficientes y relacionar correctamente los autores con sus libros.|
| **Especificación** | Crear las rutas GET /api/autores, GET /api/autores/{id} y GET /api/libros. Utilizar selectinload() o joinedload() para obtener los libros relacionados con cada autor, evitando el problema N+1. La base de datos debe incluir a Gabriel García Márquez con su libro Cien años de soledad y a Julio Cortázar con Rayuela.|
| **Output** |Entregar el archivo app.py completo y funcional, utilizando FastAPI y SQLAlchemy. Las respuestas deben estar en formato JSON, mostrar los autores con sus libros y superar las tres pruebas del verificador con HTTP 200.|

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [3]:
from verificar import probar_http

casos = [{'descripcion': 'Los autores llegan con sus libros',
  'metodo': 'GET',
  'ruta': '/api/autores',
  'estado': 200,
  'json_contiene': {'autores': [{'id': 1,
                                 'nombre': 'Gabriel García Márquez',
                                 'libros': [{'id': 1, 'titulo': 'Cien años de soledad'}]},
                                {'id': 2, 'nombre': 'Julio Cortázar', 'libros': [{'id': 2, 'titulo': 'Rayuela'}]}]}},
 {'descripcion': 'Un autor concreto trae sus libros',
  'metodo': 'GET',
  'ruta': '/api/autores/1',
  'estado': 200,
  'json_contiene': {'id': 1, 'libros': [{'id': 1, 'titulo': 'Cien años de soledad'}]}},
 {'descripcion': 'El listado de libros sigue disponible',
  'metodo': 'GET',
  'ruta': '/api/libros',
  'estado': 200,
  'json_contiene': {'libros': [{'id': 1}, {'id': 2}]}}]

probar_http(casos, archivo="app.py", limpiar=['biblioteca.db'], env={})


VERIFICACIÓN DEL RESULTADO · 3 casos · app.py
✅ 01. Los autores llegan con sus libros
✅ 02. Un autor concreto trae sus libros
✅ 03. El listado de libros sigue disponible
------------------------------------------------------------------------------
RESULTADO: 3/3 casos correctos. ¡Resultado logrado!


In [4]:
from verificar import revisar_archivo, resumen

resultados = []
resultados.append(revisar_archivo("app.py", regex=["selectinload|joinedload"], descripcion="Se usa carga ansiosa"))
resumen(resultados)


✅ Se usa carga ansiosa
------------------------------------------------------------------------------
RESULTADO: 1/1 comprobaciones correctas. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion25.md` a `auditorias/auditoria_sesion25.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   Lo que más me costó fue conseguir que los autores mostraran correctamente sus libros relacionados. Al principio tuve problemas con las consultas de la base de datos y me aparecieron errores HTTP 500. Tuve que revisar la relación entre los modelos y la manera en que se cargaba la información para poder devolver los datos correctamente.
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   Decidí utilizar SQLite para almacenar la información y organizar los datos en dos tablas relacionadas, una para autores y otra para libros. También utilicé selectinload() para cargar los libros relacionados con los autores y evitar realizar consultas innecesarias.
   - ¿Qué error detectó la verificación que tú no habías previsto?
  La verificación detectó que las rutas de autores y libros devolvían errores HTTP 500 en lugar de respuestas HTTP 200. No había considerado que una relación ORM podía presentar problemas al consultar o devolver los datos. Para solucionarlo revisé las consultas, las relaciones entre los modelos y la carga de los libros antes de generar las respuestas JSON.
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
